# Conduite — V-JEPA fidèle + System-2 (planner MPC latent)

Pipeline **fidèle à la vision LeCun** (`vjepa.py` : encodeur-contexte sur tokens visibles, prédicteur **attentionnel**, masquage par tubelets, multi-masque, SIGReg sans EMA/stop-grad).

1. **Transfert** UCF101 → dashcam Nexar (résolution H=96)
2. **Rollout collision** : on masque le futur, le World Model l'imagine, risque + heatmap
3. **System-2 (MPC latent)** : le World Model imagine les futurs sous chaque action et choisit la meilleure décision

Repo : https://github.com/frpatry/jepa-physics-bench

## 1. Récupérer le code + installer lejepa (`git pull` = dernière version)

In [ ]:
import os
if not os.path.exists('/content/jepa-physics-bench'):
    !git clone https://github.com/frpatry/jepa-physics-bench.git /content/jepa-physics-bench
!cd /content/jepa-physics-bench && git pull
%cd /content/jepa-physics-bench
!pip install -q lejepa || pip install -q git+https://github.com/rbalestr-lab/lejepa
print('OK — code à jour, lejepa installé (cv2 + huggingface_hub déjà sur Colab)')

In [ ]:
# === CACHE UCF101 via Drive : 1 archive .tar (copie SÉQUENTIELLE = rapide ; jamais d'accès aléatoire) ===
# Stratégie : 1er run -> download LOCAL + extract + tar -> upload Drive. Runtimes suivants -> copie l'archive
# du Drive en local + extrait (rapide). driving_transfer voit /content/ucf_full/_done et NE retélécharge PAS.
import os, subprocess, argparse
from google.colab import drive
drive.mount('/content/drive')
os.environ['HF_HOME'] = '/root/.cache/huggingface'             # TOUT en local (rapide). Jamais le dataset sur Drive.

DRIVE_DIR = '/content/drive/MyDrive/jepa_cache'; os.makedirs(DRIVE_DIR, exist_ok=True)
ARCHIVE = f'{DRIVE_DIR}/ucf_full.tar'                           # 1 seul gros fichier
LOCAL = '/content/ucf_full'
def sh(c): print('+', c, flush=True); subprocess.run(c, shell=True, check=True)

if os.path.isfile(f'{LOCAL}/_done'):
    print('UCF déjà prêt dans ce runtime ✅', flush=True)
elif os.path.isfile(ARCHIVE):
    print('Cache Drive trouvé -> copie séquentielle + extraction locale…', flush=True)
    sh(f'cp "{ARCHIVE}" /content/ucf_full.tar')                # Drive -> local (séquentiel = rapide, pas de hang)
    sh('tar -xf /content/ucf_full.tar -C /content')
    open(f'{LOCAL}/_done', 'w').close()
    print('UCF restauré depuis le cache Drive ✅ (zéro re-download)', flush=True)
else:
    print('Pas de cache -> téléchargement UCF101 (6 Go, UNE fois) puis archivage Drive…', flush=True)
    from lejepa_video import _download_extract
    _download_extract(argparse.Namespace(source='full'))       # download + extract -> /content/ucf_full (+_done)
    sh('tar -cf /content/ucf_full.tar -C /content ucf_full')   # 1 archive locale
    sh(f'cp /content/ucf_full.tar "{ARCHIVE}"')                # upload 1 fichier vers Drive (séquentiel)
    print('UCF téléchargé + extrait + caché sur Drive ✅ (prochains runtimes = rapides)', flush=True)

## 2. Transfert UCF101 → dashcam Nexar (H=96) — readout **attentif**
Encodeur V-JEPA entraîné sur UCF101 (générique), **gelé**, appliqué aux dashcams. On reporte **3 readouts** : linéaire / MLP (mean-pool, qui **dilue** à haute résolution) et **attentif** (requête apprise qui pondère les tokens, protocole V-JEPA — ne dilue pas).

`#2 GRAAL` = transfert cross-domaine vs `#1` in-domaine vs hasard 0.50. `--n_mask 3` = ton sweet spot multi-masque. Si **OOM** : baisser `--bs`.

In [ ]:
# H=96 + readout attentif + multi-masque 3 + plus de données
!python -u driving_transfer.py --H 96 --n_mask 3 --n_nexar 1500 --ucf_source full --ucf_nclass 50 --ucf_per 40 --bs 16

In [ ]:
# CONTRÔLE : même code à H=32 (grille 4x4) -> doit reproduire ~0.78 (ta session précédente).
# Si oui : l'archi est saine, c'était bien la dilution du mean-pool à H=96.
!python -u driving_transfer.py --H 32 --n_mask 3 --n_nexar 1500 --ucf_source full --ucf_nclass 50 --ucf_per 40 --bs 32

## 3. Prédiction de collision par rollout latent (+ heatmap) — **par transfert**
World model V-JEPA pré-entraîné sur **vidéo variée UCF101** (`--ssl_source ucf`), **gelé**, puis on transfère seulement la tête collision sur la dashcam. On masque les frames futures, le prédicteur attentionnel les **imagine**, K futurs échantillonnés → risque + GIF (image + heatmap + jauge).

⚠️ `--ucf_source full` télécharge UCF101 complet (~6 Go) au 1er run. Pour aller plus vite : `--ucf_source subset` (10 classes).

In [ ]:
!python -u driving_rollout.py --ssl_source ucf --ucf_source full --ucf_nclass 40 --ucf_per 30 --n_nexar 800 --H 96 --patch 8 --T 16 --k_viz 3 --bs 8

In [ ]:
# afficher les GIF produits par le rollout
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('/content/rollout_clip*.gif')): display(Image(p))

In [ ]:
# COMPARAISON : world model in-domaine (dashcam seule) vs varié (UCF101).
# La thèse : le world model VARIÉ anticipe-t-il mieux la collision (risque collisions >> normales) ?
!python -u driving_rollout.py --ssl_source nexar --n_nexar 800 --H 96 --patch 8 --T 16 --k_viz 0 --bs 8

## 4. System-2 — planner MPC latent (boucle fermée)
World model V-JEPA gelé → dynamique action-conditionnée `g(z,a)→z'` + tête danger `c(z)`. À chaque pas le planner imagine les futurs sous chaque frein candidat et choisit l'argmin (danger + coût).

Compare en boucle fermée : **naïf → réactif → MPC**. Surveiller la ligne `c(z) moyen : danger=… sûr=…` (doit bien séparer) et le `pred` de la dynamique (si bloqué à 0.000 → pousser `--steps` pour enrichir l'encodeur).

Compromis sécurité/confort réglable : `--w_coll` `--w_prog` `--horizon`.

In [ ]:
!python -u drive_plan.py --steps 2500 --dyn_steps 2000 --n_train 600 --n_test 150

In [ ]:
# === 5. MONDE D'OBJETS : world model qui comprend objets + dynamique (LeJEPA, sans étiquettes) ===
# Objets colorés bougent / rebondissent / se percutent. Aucun téléchargement (monde généré).
# On mesure : sait-il lire position+vitesse de chaque objet, et ANTICIPER une collision objet-objet ?
#   - SONDE ÉTAT : R² (1.0 = parfait) pour position et vitesse
#   - ANTICIPATION : world model vs oracle supervisé ("l'info est-elle là ?") vs hasard
!python -u objects.py --n 2500 --T 12 --H 48 --n_obj 3 --ssl_steps 3000 --bs 32